## OOD evaluation: GNN-only vs GINE+MLP

Runs `eval_ood_complex_voltage_checkpoints.py` on your **OOD stress** bundle and **frozen checkpoints**. Uses each checkpoint’s **training** `x_mean`/`y_mean` normalization (not recomputed on OOD).

**Prerequisite (common failure):** under `OOD_ROOT` you must have **`Heterogenous GNN dataset/nodes/hetero_mv_nodes_load_transformer_reg_tap_only.csv`** built from **OOD** OpenDSS outputs (PQ + voltages for OOD `sample_id`s). If you only ran `generate_ood_daily_aggregate_dataset_8500.py` without the printed follow-up steps, that file is missing → `FileNotFoundError`. Fix: run `aggregate_mv_node_dataset_8500.py` → `build_hetero_mv_node_type_datasets.py` → `merge_load_transformer_reg_tap_only.py --dataset-root "<OOD_ROOT>"` on the OOD folder (see the OOD generator’s footer). The script can still pick up **line edges** from the sibling `loadtype_8500_dailyagg` folder if OOD lacks the compacted edge CSV.

**Outputs:** console metrics, JSON report, optional PNGs (`--plot_dir`). Distance CSV: `8500-node/load_electrical_distance_to_each_regulator.csv` or a copy under OOD hetero.

In [ ]:
import os
import subprocess
import sys

REPO = r"C:\Users\alita\OneDrive\Desktop\GNN2"
os.chdir(REPO)

OOD_ROOT = os.path.join(REPO, r"datasets_gnn2\loadtype_8500_dailyagg\loadtype_8500_dailyagg_ood_stress")
DIST_CSV = os.path.join(REPO, r"8500-node\load_electrical_distance_to_each_regulator.csv")
GNN_ONLY_DIR = os.path.join(REPO, r"gnn2_architecture_search\homo_mv_8500\comparison\gnn_only_compare_complex_8500")
GINE_MLP_DIR = os.path.join(REPO, r"gnn2_architecture_search\homo_mv_8500\comparison\gine_plus_mlp_to_overcome_mlp_only")
OUT_JSON = os.path.join(REPO, r"gnn2_architecture_search\homo_mv_8500\comparison\ood_complex_voltage_eval.json")
PLOT_DIR = os.path.join(REPO, r"gnn2_architecture_search\homo_mv_8500\comparison\ood_eval_plots")

cmd = [
    sys.executable,
    "-u",
    "eval_ood_complex_voltage_checkpoints.py",
    "--ood_data_root",
    OOD_ROOT,
    "--electrical_distance_csv",
    DIST_CSV,
    "--eval_dirs",
    GNN_ONLY_DIR,
    GINE_MLP_DIR,
    "--batch_size",
    "8",
    "--out_json",
    OUT_JSON,
    "--plot_dir",
    PLOT_DIR,
]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(r.stdout, end="")
if r.stderr:
    print(r.stderr, end="", file=sys.stderr)
if r.returncode != 0:
    raise subprocess.CalledProcessError(r.returncode, cmd, output=r.stdout, stderr=r.stderr)
print("Report:", OUT_JSON)
print("Plots:", PLOT_DIR)